# narrativecluster — End-to-End Walkthrough

This notebook covers the full workflow:

1. **Embed** your corpus with `multilingual-e5-large-instruct`
2. **`fit()`** — build HNSW index + Leiden clusters
3. **`predict()`** — neighbour-vote cluster assignment
4. **`partial_fit()`** — add new data without re-fitting from scratch
5. **`site_stats()`** — per-platform acceptance diagnostics
6. **Save / load** a model for production reuse
7. **Export** a cluster-narrative summary

---

## 0 · Setup

In [ ]:
# !pip install narrativecluster sentence-transformers matplotlib seaborn umap-learn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

from narrativecluster import NarrativeClusterer, ClusterConfig

sns.set_theme(style='whitegrid', font_scale=1.15)
RNG = np.random.default_rng(42)

MODEL_DIR = Path('/tmp/nc_walkthrough_model')

## 1 · Embedding with multilingual-e5-large-instruct

The instruct variant of E5 needs a task-description prefix on *query* text.
For clustering, every post is a query.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = 'intfloat/multilingual-e5-large-instruct'

# ── Task instruction ─────────────────────────────────────────────────────────
# Tailor this to your data domain.  The model was trained with explicit
# instructions, so specificity helps.
E5_TASK = (
    'Given a social-media post, retrieve semantically similar posts '
    'that discuss the same narrative or claim'
)

def e5_encode(
    model: SentenceTransformer,
    texts: list[str],
    batch_size: int = 64,
    task: str = E5_TASK,
) -> np.ndarray:
    """
    Encode `texts` with the multilingual-e5-large-instruct format.

    The model requires a task-instruction prefix when encoding *queries*
    (i.e. the texts you want to cluster).  Documents added to a retrieval
    corpus would be passed without the prefix, but for clustering we treat
    every post as a query.
    """
    prefixed = [f'Instruct: {task}\nQuery: {t}' for t in texts]
    vecs = model.encode(
        prefixed,
        batch_size=batch_size,
        normalize_embeddings=True,   # cosine space
        show_progress_bar=True,
    )
    return vecs.astype(np.float32)


print(f'Loading {EMBED_MODEL_NAME} …')
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print('Done.')

### 1.1 Prepare your DataFrame

The clusterer needs three columns:

| Column | Type | Notes |
|--------|------|-------|
| `text` (or `cleaned_text`) | `str` | Deduplicated post text |
| `platform` (or `site`) | `str` | Source platform label |
| `embedding` | `np.ndarray` shape `(D,)` | Output of `e5_encode` |

In [ ]:
# ── Option A: your real data ──────────────────────────────────────────────────
# df_raw = pd.read_parquet('annotation_inputs/df_claim_level.parquet')
# vecs = e5_encode(embed_model, df_raw['cleaned_text'].tolist())
# df_raw['embedding'] = list(vecs)

# ── Option B: synthetic demo data ────────────────────────────────────────────
# We generate three waves:
#   • df_train  : initial fit (4000 posts)
#   • df_new    : partial_fit batch (1000 new posts)
#   • df_unseen : predict-only batch (500 posts, no Leiden membership)

NARRATIVES = {
    'election_fraud':  ['ballot stuffing', 'rigged machines', 'stolen election',
                        'mail-in fraud', 'fake ballots', 'voter fraud'],
    'vaccine_harm':    ['mRNA danger', 'spike protein', 'vaccine mandate',
                        'natural immunity', 'myocarditis', 'booster harm'],
    'immigration':     ['open borders', 'illegal crossings', 'caravan threat',
                        'deportation', 'asylum fraud', 'border wall'],
    'climate_hoax':    ['carbon tax', 'fossil fuel lies', 'global warming myth',
                        'green agenda', 'arctic ice', 'net zero scam'],
    'media_censorship':['fake news', 'shadow ban', 'big tech censorship',
                        'fact-check bias', 'de-platform', 'propaganda'],
}
PLATFORMS = ['twitter', 'telegram', 'gab', '4chan', 'truthsocial']
TEMPLATES = [
    'BREAKING: {a} exposed! Share before deletion.',
    'The truth about {a} they hide from you.',
    'New evidence: {a} confirmed by whistleblowers.',
    'Why is {a} being suppressed?',
    'MUST READ: {a} is worse than reported.',
    'Wake up — {a} is real and happening now.',
    'Just posted: {a} — {a} everywhere.',
]

def make_wave(n_per_narrative: int, noise_n: int, seed: int, text_prefix: str = '') -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    for nar, anchors in NARRATIVES.items():
        for _ in range(n_per_narrative):
            a = rng.choice(anchors)
            t = rng.choice(TEMPLATES).format(a=a)
            p = rng.choice(PLATFORMS, p=[.38, .27, .15, .12, .08])
            rows.append({'text': text_prefix + t, 'platform': p, 'true_narrative': nar})
    noise_words = ['weather', 'recipe', 'sports', 'local event', 'movie', 'pet', 'travel']
    for _ in range(noise_n):
        w = rng.choice(noise_words)
        rows.append({'text': text_prefix + f'Just saw something about {w} today!',
                     'platform': rng.choice(PLATFORMS), 'true_narrative': 'noise'})
    df = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

df_train   = make_wave(n_per_narrative=600, noise_n=200, seed=0)
df_new     = make_wave(n_per_narrative=150, noise_n=50,  seed=1, text_prefix='[new] ')
df_unseen  = make_wave(n_per_narrative=80,  noise_n=20,  seed=2, text_prefix='[unseen] ')

print(f'Train: {len(df_train):,}  |  New batch: {len(df_new):,}  |  Unseen: {len(df_unseen):,}')

In [ ]:
# Embed all three waves
for name, df in [('train', df_train), ('new', df_new), ('unseen', df_unseen)]:
    print(f'Encoding {name} …')
    vecs = e5_encode(embed_model, df['text'].tolist())
    df['embedding'] = list(vecs)

print('\nEmbedding shapes:')
for name, df in [('train', df_train), ('new', df_new), ('unseen', df_unseen)]:
    print(f'  {name}: {np.vstack(df["embedding"].values).shape}')

## 2 · Configure and fit

In [ ]:
cfg = ClusterConfig(
    # Column names
    embed_col='embedding',
    text_col='text',
    site_col='platform',

    # HNSW index
    hnsw_m=32,
    ef_construction=200,
    ef_search=128,

    # Graph / Leiden
    k_graph=32,
    tau_graph=0.65,       # cosine-sim threshold for Leiden graph edges
    cap_m=16,             # max out-degree per node
    leiden_resolution=1.0,

    # Neighbour-vote prediction
    # 👉 See notebook 01_tau_edge_tuning.ipynb to choose tau_edge
    kq=40,
    tau_edge=0.68,
    min_cluster_size=4,
    vote_margin=2,

    # Runtime
    query_chunk=1_000,
    random_seed=42,
)

model = NarrativeClusterer(config=cfg, checkpoint_dir=MODEL_DIR)
model.fit(df_train)

In [ ]:
# Cluster size distribution
diag = model.cluster_diagnostics()
print(f'Total clusters   : {len(diag)}')
print(f'Eligible (≥4 posts): {diag["eligible"].sum()}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(diag['size'], bins=40, color='steelblue', edgecolor='white')
ax.set_xlabel('Cluster size (# posts)')
ax.set_ylabel('Count')
ax.set_title('Leiden cluster size distribution')
plt.tight_layout()
plt.show()

## 3 · Predict — neighbour-vote assignment

In [ ]:
df_pred = model.predict(df_train)

print('Prediction columns added:')
pred_cols = [c for c in df_pred.columns if c.startswith('nv_')]
print(pred_cols)

acc_rate = df_pred['nv_accepted'].mean()
print(f'\nAcceptance rate: {acc_rate:.1%}')
print(f'Posts assigned : {df_pred["nv_accepted"].sum():,} / {len(df_pred):,}')

In [ ]:
# Distribution of prediction confidence
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_acc = df_pred[df_pred['nv_accepted']]

axes[0].hist(df_acc['nv_vote_frac'], bins=30, color='#2196F3', edgecolor='white')
axes[0].set_title('Vote fraction');  axes[0].set_xlabel('nv_vote_frac')

axes[1].hist(df_acc['nv_votes_tot'], bins=20, color='#FF9800', edgecolor='white')
axes[1].set_title('Total votes');  axes[1].set_xlabel('nv_votes_tot')

axes[2].hist(df_acc['nv_best_sim'], bins=30, color='#4CAF50', edgecolor='white')
axes[2].set_title('Best-neighbour sim');  axes[2].set_xlabel('nv_best_sim')

plt.suptitle('Prediction confidence (accepted posts)', fontweight='bold')
plt.tight_layout()
plt.show()

## 4 · Platform / site diagnostics

In [ ]:
global_rate, site_tbl = model.site_stats(df_pred)
print(f'Global acceptance rate: {global_rate:.1%}\n')
site_tbl[['count', 'accept_rate', 'abs_diff_from_global', 'cohens_h', 'effect_size']]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#E91E63' if abs(h) >= 0.2 else '#90A4AE'
          for h in site_tbl['cohens_h'].values]
bars = ax.barh(site_tbl.index, site_tbl['accept_rate'], color=colors)
ax.axvline(global_rate, color='black', linestyle='--', label=f'Global: {global_rate:.1%}')
ax.set_xlabel('Acceptance rate');  ax.set_title('Per-platform acceptance (pink = |Cohen h| ≥ 0.2)')
ax.legend()
plt.tight_layout()
plt.show()

## 5 · Visualise clusters with UMAP

A 2-D projection is useful for a sanity check — do the Leiden clusters look
geometrically coherent?

In [ ]:
try:
    import umap
    HAS_UMAP = True
except ImportError:
    print('umap-learn not installed.  Run: pip install umap-learn')
    HAS_UMAP = False

if HAS_UMAP:
    # Project a sample of accepted posts
    sample = df_pred[df_pred['nv_accepted']].sample(min(2000, df_pred['nv_accepted'].sum()), random_state=42)
    X = np.vstack(sample['embedding'].values).astype(np.float32)

    reducer = umap.UMAP(n_components=2, metric='cosine', random_state=42, verbose=False)
    xy = reducer.fit_transform(X)

    # Colour by true narrative (ground-truth label)
    nar_vals = sample['true_narrative'].values
    unique_nar = list(pd.Series(nar_vals).unique())
    palette = sns.color_palette('tab10', len(unique_nar))
    color_map = {n: palette[i] for i, n in enumerate(unique_nar)}
    colors = [color_map[n] for n in nar_vals]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Left: coloured by true narrative
    for nar in unique_nar:
        m = nar_vals == nar
        axes[0].scatter(xy[m, 0], xy[m, 1], s=8, alpha=0.55,
                        color=color_map[nar], label=nar)
    axes[0].set_title('UMAP — coloured by TRUE narrative')
    axes[0].legend(fontsize=8, markerscale=2)

    # Right: coloured by predicted Leiden cluster (top-15 largest only)
    top_clusters = sample['nv_pred_cluster'].value_counts().head(15).index.tolist()
    pal2 = sns.color_palette('husl', len(top_clusters))
    c2_map = {c: pal2[i] for i, c in enumerate(top_clusters)}
    c_vals = sample['nv_pred_cluster'].values
    colors2 = [c2_map.get(c, (0.7, 0.7, 0.7)) for c in c_vals]
    axes[1].scatter(xy[:, 0], xy[:, 1], s=8, alpha=0.55, c=colors2)
    axes[1].set_title('UMAP — coloured by PREDICTED cluster (top-15)')

    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    plt.savefig('/tmp/umap_clusters.png', dpi=150)
    plt.show()

## 6 · Incremental update with `partial_fit`

Suppose a new batch of posts arrives (e.g. next week's data).
Instead of re-fitting from scratch, we call `partial_fit()`.

In [ ]:
print(f'Before partial_fit: {model.n_samples_fit_:,} rows, {model.n_clusters_} clusters')

model.partial_fit(df_new)

print(f'After  partial_fit: {model.n_samples_fit_:,} rows, {model.n_clusters_} clusters')

In [ ]:
# Predict on the new batch to see how many slots in
df_new_pred = model.predict(df_new)
print(f'New batch acceptance rate: {df_new_pred["nv_accepted"].mean():.1%}')

# And on unseen posts that were never in the training set
df_unseen_pred = model.predict(df_unseen)
print(f'Unseen acceptance rate:    {df_unseen_pred["nv_accepted"].mean():.1%}')

### How partial_fit handles embedding space stability

New vectors are **mean-centred using the original µ** (computed during `fit()`).
This keeps the HNSW space geometrically stable — new posts land in the same
neighbourhood structure as the original corpus.  The tradeoff is that if the
new data has a very different average embedding (e.g. a totally different domain),
you may want to call `fit()` again from scratch.

In [ ]:
# Visualise shift in the mean
original_mu = model.mu_.squeeze()

new_vecs = np.vstack(df_new['embedding'].values)
new_mu_would_be = new_vecs.mean(axis=0)

cosine_drift = float(
    np.dot(original_mu, new_mu_would_be)
    / (np.linalg.norm(original_mu) * np.linalg.norm(new_mu_would_be) + 1e-12)
)
print(f'Cosine similarity between train µ and new-batch µ: {cosine_drift:.4f}')
print('(≥ 0.99 → safe to reuse µ;  < 0.95 → consider full refit)')

## 7 · Save and reload the model

In [ ]:
SAVE_PATH = Path('/tmp/nc_walkthrough_saved')
model.save(SAVE_PATH)
print(f'Saved to {SAVE_PATH}')
print('Files:')
for f in sorted(SAVE_PATH.iterdir()):
    print(f'  {f.name:40s}  {f.stat().st_size / 1024:.1f} KB')

In [ ]:
model2 = NarrativeClusterer.load(SAVE_PATH, config=cfg)
print(f'Reloaded — {model2.n_samples_fit_:,} rows, {model2.n_clusters_} clusters')

# Verify predictions are identical
df_check = model2.predict(df_unseen)
assert (df_check['nv_accepted'] == df_unseen_pred['nv_accepted']).all(), 'Mismatch!'
print('✅  Loaded model produces identical predictions.')

## 8 · Export a cluster narrative summary

For each predicted cluster, pull the top-3 most central posts (highest
`nv_best_sim`) as a human-readable summary.

In [ ]:
df_all = model.predict(model.df_unique_)  # predict on entire fitted corpus

accepted = df_all[df_all['nv_accepted']].copy()

summary_rows = []
for cid, grp in accepted.groupby('nv_pred_cluster'):
    top = grp.nlargest(3, 'nv_best_sim')[['text', 'platform', 'nv_best_sim', 'nv_vote_frac']]
    summary_rows.append({
        'cluster_id':   cid,
        'n_posts':      len(grp),
        'platforms':    ', '.join(sorted(grp['platform'].unique())),
        'top_post_1':   top.iloc[0]['text'] if len(top) > 0 else '',
        'top_post_2':   top.iloc[1]['text'] if len(top) > 1 else '',
        'top_post_3':   top.iloc[2]['text'] if len(top) > 2 else '',
        'mean_vote_frac': grp['nv_vote_frac'].mean(),
    })

cluster_summary = (
    pd.DataFrame(summary_rows)
      .sort_values('n_posts', ascending=False)
      .reset_index(drop=True)
)

cluster_summary.head(10)

In [ ]:
# Save to CSV for annotation / manual review
OUT = Path('/tmp/cluster_summary.csv')
cluster_summary.to_csv(OUT, index=False)
print(f'Saved cluster summary: {OUT}  ({len(cluster_summary)} clusters)')

## 9 · Common gotchas & tips

### Gotcha 1 — tau_graph vs tau_edge

`tau_graph` controls which edges enter the **Leiden graph** (built once at `fit()` time).  
`tau_edge` controls which neighbours are allowed to cast votes at **predict()** time.  
They are independent.  Lowering `tau_edge` increases coverage; raising it increases precision.  
**→ See `01_tau_edge_tuning.ipynb` for a data-driven way to set `tau_edge`.**

### Gotcha 2 — E5 cosine similarities are already high

`multilingual-e5-large-instruct` outputs L2-normalised vectors that tend to
have higher cosine similarities (0.7–0.9) than older sentence models (0.5–0.8).
This means sensible defaults are higher than for, say, `all-mpnet-base-v2`.
Start with `tau_graph ≈ 0.65` and `tau_edge ≈ 0.68`.

### Gotcha 3 — Cluster IDs shift after `partial_fit`

Leiden is re-run on the full merged graph after each `partial_fit`, so cluster
integer IDs can change.  If you need stable IDs across incremental updates,
build a mapping from the old `leiden_cluster` column before calling `partial_fit`.

### Gotcha 4 — Memory on large corpora

For >5M posts the dense `I_knn` / `D_knn` arrays (shape `N × K`) can exceed
RAM.  Use `np.save` / `mmap_mode='r'` (already done by the package) and keep
`k_graph` ≤ 32.

### Tip — batching `partial_fit` for streaming data

```python
# Process weekly batches
for week_df in weekly_batches:
    vecs = e5_encode(embed_model, week_df['text'].tolist())
    week_df['embedding'] = list(vecs)
    model.partial_fit(week_df)
    
# Save after each batch
model.save('/data/models/narrative_model_latest')
```